# Word Weight Comparison

This notebook compares the final word weights learned by Perceptron, Average Perceptron, and Pegasos on the **same train/validation/test split and vocabulary** used by `automatic_review_analyzer.ipynb`. Pegasos uses the selected `lambda*` from validation, so this notebook analyzes the final models rather than a separate Pegasos run.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'project_1':
    candidate = PROJECT_DIR / 'unit_1' / 'project_1'
    if candidate.exists(): PROJECT_DIR = candidate
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
from linear_classification import accuracy, average_perceptron, pegasos, perceptron
from review_data import prepare_review_data

## 1. Use the project-wide experimental split

The shared review-data module performs the reproducible stratified 70/15/15 split and builds the vocabulary from training reviews only. This prevents this notebook from silently training on a different subset than the main experiment.

In [ ]:
SEED = 42
EPOCHS = 10
BATCH_SIZE = 32
data = prepare_review_data(PROJECT_DIR, seed=SEED, min_count=2)
vocabulary = data['vocabulary']
X_train, y_train = data['X_train'], data['y_train']
X_validation, y_validation = data['X_validation'], data['y_validation']
train_data = list(zip(y_train, X_train))
validation_data = list(zip(y_validation, X_validation))
final_train_data = train_data + validation_data
print(f'Train: {len(train_data)} | Validation: {len(validation_data)} | Final training set: {len(final_train_data)}')
print(f'Vocabulary size: {len(vocabulary)}')

## 2. Use the selected Pegasos `lambda*`

The main notebook selects Pegasos regularization by maximum validation accuracy. To keep this analysis reproducible and consistent with that experiment, the selected value is recorded explicitly here.

$$
\lambda^*=\arg\max_{\lambda}\mathrm{ValidationAccuracy}(\lambda).
$$

In [ ]:
SELECTED_LAMBDA = 5e-3
print(f'Using selected lambda*: {SELECTED_LAMBDA:.1e}')

models = {
    'Perceptron': perceptron(final_train_data, epochs=EPOCHS),
    'Average Perceptron': average_perceptron(final_train_data, epochs=EPOCHS),
    'Pegasos': pegasos(final_train_data, lambda_=SELECTED_LAMBDA, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED),
}
for name, weights in models.items():
    print(f'{name:20} | train+validation accuracy={accuracy(weights, final_train_data):.2%}')

## 3. Strongest learned words

For a word feature `j`, its contribution to a review score is `theta_j * x_j`. With binary bag-of-words, a present word has `x_j = 1`, so its weight is directly the contribution of that word. Positive weights favor the positive class; negative weights favor the negative class.

In [ ]:
index_to_word = {index: word for word, index in vocabulary.items()}
for name, weights in models.items():
    positive = sorted(((weight, index_to_word[index]) for index, weight in weights.items() if weight > 0), reverse=True)[:15]
    negative = sorted(((weight, index_to_word[index]) for index, weight in weights.items() if weight < 0))[:15]
    print(f'\n{name}')
    print('Strongest positive words:', [(word, round(weight, 4)) for weight, word in positive])
    print('Strongest negative words:', [(word, round(weight, 4)) for weight, word in negative])

In [ ]:
common_words = set()
for weights in models.values():
    common_words.update(index_to_word[index] for index in sorted(weights, key=lambda index: abs(weights[index]), reverse=True)[:20])
words = sorted(common_words)
indices = np.arange(len(words))
width = 0.25
fig, ax = plt.subplots(figsize=(14, 6))
for offset, (name, weights) in zip((-width, 0, width), models.items()):
    ax.bar(indices + offset, [weights.get(vocabulary[word], 0.0) for word in words], width, label=name)
ax.axhline(0, linewidth=0.8)
ax.set_xticks(indices); ax.set_xticklabels(words, rotation=60, ha='right')
ax.set_ylabel('Learned weight'); ax.set_title('Final learned word weights across the three classifiers'); ax.legend()
plt.tight_layout(); plt.show()

## 4. Interpretation

All three classifiers learn a parameter for every vocabulary feature. Pegasos is different because its regularized objective controls the size of the parameter vector, so its weights can be much smaller than the unregularized Perceptron and Average Perceptron weights.

The important comparison is therefore not simply which classifier has the largest numerical weight. The learned weights belong to models with different optimization rules and objectives.

Dataset source: Kotzias et al., **Sentiment Labelled Sentences**, UCI Machine Learning Repository, DOI 10.24432/C57604.